In [1]:
# Computing normal modes and solving linear systems for vibration and statics problems

import numpy as np
from scipy.linalg import eigh

def compute_two_mass_system(m1, m2, k1, k2, k3):
    # Mass and stiffness matrices
    M = np.diag([m1, m2])
    K = np.array([[k1 + k2, -k2],
                  [-k2, k2 + k3]])
    # Solve generalized eigenvalue problem
    w2, evecs = eigh(K, M)
    freqs = np.sqrt(w2)
    return freqs, evecs

def count_nodes(evec):
    # Count number of sign changes (zero crossings) in eigenvector
    return np.sum(np.diff(np.sign(evec)) != 0)

# Part 1: Two-mass system
results_part1 = {}

# (a) Symmetric case
freqs_a, evecs_a = compute_two_mass_system(1.0, 1.0, 1.0, 1.0, 1.0)
results_part1['a'] = (freqs_a, evecs_a)

# (b) m2 = 2.0
freqs_b, evecs_b = compute_two_mass_system(1.0, 2.0, 1.0, 1.0, 1.0)
results_part1['b'] = (freqs_b, evecs_b)

# (c) k2 = 0.05
freqs_c, evecs_c = compute_two_mass_system(1.0, 1.0, 1.0, 0.05, 1.0)
results_part1['c'] = (freqs_c, evecs_c)

# (d) Double all spring constants
freqs_d, evecs_d = compute_two_mass_system(1.0, 1.0, 2.0, 2.0, 2.0)
results_part1['d'] = (freqs_d, evecs_d)

# Part 2: N-mass chain
N = 10
m = 1.0
k = 1.0

# Mass matrix
M_N = np.eye(N) * m

# Stiffness matrix K
K_N = np.zeros((N, N))
for i in range(N):
    if i > 0:
        K_N[i, i] += k
        K_N[i, i-1] -= k
    if i < N - 1:
        K_N[i, i] += k
        K_N[i, i+1] -= k

# Solve generalized eigenvalue problem
w2_N, evecs_N = eigh(K_N, M_N)
freqs_N = np.sqrt(w2_N)

# Count nodes in first 4 and 10th mode
node_counts = {}
for mode in [0, 1, 2, 3, 9]:
    nodes = count_nodes(evecs_N[:, mode])
    node_counts[f"Mode {mode+1}"] = nodes

# Part 3: Array indexing
evecs = evecs_N  # shape (10, 10)
first_mass_all_modes = evecs[0]
first_mode_all_masses = evecs[:, 0]
mass3_mode5 = evecs[2, 4]

# Part 4: Linear algebra examples

# (a) Kirchhoff loop rule
R1, R2, R3 = 10, 20, 30
E1, E2 = 5, 10
A_kirchhoff = np.array([[R1 + R2, -R2, 0],
                        [-R2, R2 + R3, -R3],
                        [1, 1, -1]])
b_kirchhoff = np.array([E1, E2, 0])
I_kirchhoff = np.linalg.solve(A_kirchhoff, b_kirchhoff)

# (b) Static equilibrium
L = 4.0
W = 100.0
d = 1.5
A_static = np.array([[1, 1],
                     [0, L]])
b_static = np.array([W, W * d])
R_static = np.linalg.solve(A_static, b_static)

# Print results
print("Part 1: Two-mass system results")
for key in ['a', 'b', 'c', 'd']:
    freqs, evecs = results_part1[key]
    print(f"\nCase {key}:")
    print("Frequencies (rad/s):", np.round(freqs, 4))
    print("Eigenvectors (columns are modes):\n", np.round(evecs, 4))

print("\nPart 2: N-mass chain (N=10)")
print("Frequencies (rad/s):", np.round(freqs_N, 4))
print("Node counts for selected modes:")
for mode, count in node_counts.items():
    print(f"  {mode}: {count} nodes")
print("Eigenvector for 10th mode:\n", np.round(evecs_N[:, 9], 4))

print("\nPart 3: Array indexing")
print("evecs[0] (first mass across all modes):\n", np.round(first_mass_all_modes, 4))
print("evecs[:, 0] (first mode across all masses):\n", np.round(first_mode_all_masses, 4))
print("Displacement of Mass #3 in Mode #5 (evecs[2, 4]):", round(mass3_mode5, 4))

print("\nPart 4: Linear algebra examples")
print("Kirchhoff loop rule solution (currents I1, I2, I3):", np.round(I_kirchhoff, 4))
print("Static equilibrium reactions (RA, RB):", np.round(R_static, 4))


Part 1: Two-mass system results

Case a:
Frequencies (rad/s): [1.     1.7321]
Eigenvectors (columns are modes):
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]

Case b:
Frequencies (rad/s): [0.7962 1.5382]
Eigenvectors (columns are modes):
 [[-0.4597 -0.8881]
 [-0.628   0.3251]]

Case c:
Frequencies (rad/s): [1.     1.0488]
Eigenvectors (columns are modes):
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]

Case d:
Frequencies (rad/s): [1.4142 2.4495]
Eigenvectors (columns are modes):
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]

Part 2: N-mass chain (N=10)
Frequencies (rad/s): [   nan 0.3129 0.618  0.908  1.1756 1.4142 1.618  1.782  1.9021 1.9754]
Node counts for selected modes:
  Mode 1: 0 nodes
  Mode 2: 1 nodes
  Mode 3: 2 nodes
  Mode 4: 3 nodes
  Mode 10: 9 nodes
Eigenvector for 10th mode:
 [-0.07    0.203  -0.3162  0.3985 -0.4417  0.4417 -0.3985  0.3162 -0.203
  0.07  ]

Part 3: Array indexing
evecs[0] (first mass across all modes):
 [ 0.3162 -0.4417  0.4253 -0.3985 -0.3618 -0.3162 -0.2629 -0.203   0.

C:\Users\17069\AppData\Local\Temp\ipykernel_12352\1463233165.py:59: RuntimeWarning: invalid value encountered in sqrt
  freqs_N = np.sqrt(w2_N)


Note: copilot was used to debug code